## DLA CODE OF CONDUCT V2.0

This Code of Conduct defines the principles governing ethical, transparent, and responsible use of Large Language Models (LLMs), online resources, and peer collaboration in the Deep Learning Applications laboratories. This version of the Code of Conduct was refined via a brainstorming session with **ChatGPT Version 5.2** and subsequently adapted to reflect the specific requirements and values of the DLA laboratories. In that spirit, this Code itself models the transparency it expects from you.

***Our goal is not to restrict innovation, but to ensure integrity, accountability, and genuine learning.***

### 1. Transparency in the Use of LLMs and AI Tools

The use of LLMs and AI-assisted tools is permitted — *but it must be transparent*.

* **Explicit Disclosure:** Clearly state if and how LLMs (e.g., ChatGPT, Copilot, Claude, etc.) were used. This includes code generation, debugging, data analysis, experiment design, report writing, or conceptual clarification.
* **Description of Contribution:** Briefly describe what the tool contributed and how you modified, verified, or extended its output.
* **Acknowledgment of Limitations:** Recognize that LLM outputs may contain errors, biases, or non-optimal solutions. You are responsible for verifying correctness, appropriateness, and academic integrity.

***Using AI does not reduce your responsibility for the final result.***

### 2. Proper Attribution and Documentation

Deep learning builds on existing work — responsibly.

* **Attribution:** Properly cite all external resources, including: Code snippets, Tutorials, Documentation, Datasets, Pretrained models, Research papers, and AI-generated content.
* **Reproducibility:** Clearly document tools, libraries, model versions, hyperparameters, and experimental setups so that your work can be reproduced.
* **Clarity of Modifications:** If you adapt external code, explicitly indicate what you changed and why.

***Transparency is a sign of scientific maturity — not weakness.***

### 3. Collaboration and Individual Responsibility

Discussion is encouraged. Copying is not.

* **Collaborative Learning:** You are encouraged to discuss concepts, debugging strategies, and approaches with classmates.
* **Individual Submission:** Your submitted solution must reflect your own understanding and implementation.
* **No Direct Sharing of Solutions:** Do not share complete solutions, trained models, or reports. Do not submit another person's work — or AI-generated work — as your own without meaningful engagement and proper disclosure.

***If you cannot explain your submission, it is not your submission.***

### 4. Accountability and Academic Integrity

You are responsible for everything you submit. Failure to comply with these guidelines may result in review by the course examination commission and can lead to disciplinary measures in accordance with university regulations.

***Integrity is part of your training as a machine learning practitioner.***

### 5. The Spirit of This Code of Conduct

This course prepares you to work in a field where:

* Reproducibility matters
* Ethical considerations matter
* Transparency matters
* Responsible AI use matters

***The purpose of this Code of Conduct is not surveillance — it is professional formation.***

### TL;DR

Use AI; Don’t let AI use you; Be transparent; Cite everything; Do your own thinking.

***If you can’t explain it, you probably shouldn’t submit it.***

---
---

## Introduction

In this second laboratory we will gain some experience working with Transformer models for a variety tasks using (mostly) the Hugging Face Ecosystem. 


---
### Exercise 1: Sentiment Analysis (warm up)

In this first exercise we will start from a pre-trained BERT transformer and build up a model able to perform text sentiment analysis. Transformers are complex beasts, so we will build up our pipeline in several explorative and incremental steps.

#### Exercise 1.1: Loading the Dataset Splits
There are a many sentiment analysis datasets, but we will use one of the smallest ones available: the [Cornell Rotten Tomatoes movie review dataset](https://huggingface.co/datasets/cornell-movie-review-data/rotten_tomatoes), which consists of 5,331 positive and 5,331 negative processed sentences from the Rotten Tomatoes movie reviews.

**Your first task**: Load the dataset and figure out what splits are available and how to get them. Spend some time exploring the dataset to see how it is organized. Note that we will be using the [HuggingFace Datasets](https://huggingface.co/docs/datasets/en/index) library for downloading, accessing, splitting, and batching data for training and evaluation.

In [2]:
import torch
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [3]:
from datasets import load_dataset, get_dataset_split_names,load_dataset_builder

ds_builder = load_dataset_builder("cornell-movie-review-data/rotten_tomatoes")

We get every information possible here without downloading it

In [4]:
ds_builder.info

DatasetInfo(features={'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}, builder_name='parquet', dataset_name='rotten_tomatoes', config_name='default', version=0.0.0, splits={'train': SplitInfo(name='train', num_bytes=1075873, num_examples=8530, dataset_name='rotten_tomatoes'), 'validation': SplitInfo(name='validation', num_bytes=134809, num_examples=1066, dataset_name='rotten_tomatoes'), 'test': SplitInfo(name='test', num_bytes=136102, num_examples=1066, dataset_name='rotten_tomatoes')}, download_size=881052, dataset_size=1346784, size_in_bytes=2227836)

In [5]:
print("Features description:\n",ds_builder.info.features)

Features description:
 {'text': Value('string'), 'label': ClassLabel(names=['neg', 'pos'])}


Now we load our dataset consisting of all three splits

In [6]:
dataset = load_dataset("cornell-movie-review-data/rotten_tomatoes")
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 8530
    })
    validation: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 1066
    })
})

In [7]:
ds_train = dataset["train"]
ds_val = dataset["validation"]
ds_test = dataset["test"]

Now let's inspect ds_train object

In [8]:
import numpy as np
indices = np.random.choice(len(ds_train), 6, replace=False)
print("The following example are structure as follows \n <label> : <phrase>\n")
for index in indices:
    print(ds_train[index]["label"],":",ds_train[index]["text"])

The following example are structure as follows 
 <label> : <phrase>

0 : the rock has a great presence but one battle after another is not the same as one battle followed by killer cgi effects .
1 : an effective portrait of a life in stasis -- of the power of inertia to arrest development in a dead-end existence .
1 : a fun ride .
1 : rouge is less about a superficial midlife crisis than it is about the need to stay in touch with your own skin , at 18 or 80 .
1 : unlike the speedy wham-bam effect of most hollywood offerings , character development -- and more importantly , character empathy -- is at the heart of italian for beginners .
0 : there's a whole heap of nothing at the core of this slight coming-of-age/coming-out tale .



---
### Exercise 1.2: A Pre-trained BERT and Tokenizer

The model we will use is a *very* small BERT transformer called [DistilBERT](https://huggingface.co/distilbert/distilbert-base-uncased) this model was trained (using self-supervised learning) on the same corpus as BERT but using the full BERT base model as a *teacher*.

**Your next task**: Load the DistilBERT model and corresponding tokenizer. Use the tokenizer on a few samples from the dataset and pass the tokens through the model to see what outputs are provided. I suggest you use the [`AutoModel`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html) class (and the `from_pretrained()` method) to load the model and `AutoTokenizer` to load the tokenizer).

In [9]:
# AutoClass imports.
from transformers import AutoTokenizer, AutoModel

model = AutoModel.from_pretrained('distilbert/distilbert-base-uncased')
tokenizer = AutoTokenizer.from_pretrained('distilbert/distilbert-base-uncased')


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
model

DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSelfAttention(
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=Tru

In [11]:
type(tokenizer)

transformers.models.bert.tokenization_bert.BertTokenizer

In [12]:
tokenized_sentence = tokenizer.encode(ds_train[0]["text"])
len(tokenized_sentence)

## now let's try to tokenize more than one frase
tokens = tokenizer.encode([ds_train[0]["text"],ds_train[1]["text"]])
len(tokens)

2

Let's verify the lenght of each sentence

In [13]:
for token in tokens :
    print(len(token))

47
52


We also can get the tensor,but when dealing with tensor we have to use padding

In [14]:
tokenized_sentence = tokenizer.encode([ds_train[0]["text"],ds_train[1]["text"]],return_tensors='pt',padding=True)
tokenized_sentence.shape

torch.Size([2, 52])

Now we can feed the model our tokenized phrases but first let's create a function to actually collect pharases.

In [15]:
def collect_tokenized_tensor(phrases: list[str] | str, tokenizer, num_sample=None):
    if isinstance(phrases, str):
        phrases = [phrases]

    if num_sample is not None and num_sample < len(phrases):
        indices = np.random.choice(len(phrases), num_sample, replace=False)
        phrases = [phrases[i] for i in indices]

    return tokenizer.encode(phrases,return_tensors="pt",padding=True),indices

In [16]:
input_tensors , indices  = collect_tokenized_tensor(ds_train["text"],tokenizer,num_sample= 2)
print(input_tensors.shape)
indices

torch.Size([2, 27])


array([8123, 6213])

In [17]:
out = model(input_tensors)
dir(out)
type(out)

transformers.modeling_outputs.BaseModelOutput

In [18]:
out.last_hidden_state.shape

torch.Size([2, 27, 768])

In [19]:
batch = tokenizer(ds_train[indices]['text'], return_tensors='pt', padding=True)
batch

{'input_ids': tensor([[  101,  1996,  2143,  2003, 12781,  2091,  2011,  4637,  3494,  2040,
          2024,  2593,  2205,  2204,  2135,  1010,  7968,  1998,  4209,  2030,
          2091, 15950, 29257,  2135,  4763,  1012,   102],
        [  101,  2009,  1005,  1055, 10468,  2019,  2058, 10052,  2792,  1997,
          7122,  2013,  1996, 19888,  1012,   102,     0,     0,     0,     0,
             0,     0,     0,     0,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1],
        [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0]])}

Attention mask tells the model that it does not have to consider those tokens 

In [20]:
print(type(batch))
tokenizer.decode(token_ids = batch['input_ids'])

<class 'transformers.tokenization_utils_base.BatchEncoding'>


['[CLS] the film is weighed down by supporting characters who are either too goodly, wise and knowing or downright comically evil. [SEP]',
 "[CLS] it ' s basically an overlong episode of tales from the crypt. [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]"]

In [21]:
no_masking = model.forward(input_ids=batch['input_ids']).last_hidden_state
print(no_masking)
assert torch.allclose(no_masking, out.last_hidden_state, atol=1e-5), "Tensor are NOT equal"

masking = model.forward(input_ids=batch['input_ids'], attention_mask=batch['attention_mask']).last_hidden_state
print(torch.allclose(masking, out.last_hidden_state, atol=1e-5))

## equivantly i can use
_masking = model.forward(**batch).last_hidden_state
print(torch.allclose(_masking, out.last_hidden_state, atol=1e-5))


tensor([[[-0.0600, -0.0413, -0.0130,  ..., -0.0799,  0.4698,  0.3896],
         [ 0.0947, -0.1917, -0.3356,  ...,  0.1952,  1.0380, -0.0909],
         [ 0.3356,  0.0928, -0.2264,  ..., -0.0238,  0.2310,  0.0462],
         ...,
         [ 0.0360,  0.3972,  0.2642,  ..., -0.1939,  0.2315, -0.5623],
         [ 0.8629,  0.1623, -0.2368,  ...,  0.2085, -0.3155, -0.3273],
         [ 0.7046,  0.4523, -0.0129,  ...,  0.2506, -0.2658, -0.0938]],

        [[-0.1447, -0.4502,  0.1316,  ...,  0.2345,  0.6385,  0.3204],
         [ 0.0470, -0.7359,  0.0355,  ...,  0.3286,  0.5982,  0.0347],
         [ 0.2278, -0.6805,  0.5542,  ..., -0.0398,  0.5271, -0.0878],
         ...,
         [ 0.2893, -0.2217,  0.4421,  ...,  0.0577,  0.4005, -0.3944],
         [ 0.3003, -0.2424,  0.4645,  ...,  0.0703,  0.3806, -0.4550],
         [ 0.2944, -0.2569,  0.4726,  ...,  0.0892,  0.3805, -0.4875]]],
       grad_fn=<NativeLayerNormBackward0>)
False
False



---
### Exercise 1.3: A Stable Baseline

In this exercise I want you to:
1. Use DistilBERT as a *feature extractor* to extract representations of the text strings from the dataset splits;
2. Train a classifier (your choice, by an SVM from Scikit-learn is an easy choice).
3. Evaluate performance on the validation and test splits.

These results are our *stable baseline* -- the **starting** point on which we will (hopefully) improve in the next exercise.

**Hint**: There are a number of ways to implement the feature extractor, but probably the best is to use a [feature extraction `pipeline`](https://huggingface.co/tasks/feature-extraction). You will need to interpret the output of the pipeline and extract only the `[CLS]` token from the *last* transformer layer. *How can you figure out which output that is?*

In [ ]:
from transformers import pipeline
import torch
from sklearn.svm import SVC
from sklearn.metrics import classification_report

def  extract_features(model,tokenizer,device,ds,batch_size=128):
    feature_extractor = pipeline("feature-extraction",framework="pt", model=model, tokenizer=tokenizer, 
                                device=device)
    feats = feature_extractor(list(ds["text"]), return_tensors='pt', batch_size=batch_size)
    cls_feats = torch.vstack([feats[i][0][0] for i in range(len(feats))])
    return cls_feats

Now we use the svm to classify

In [37]:
## feature extractions
cls_feats_train = extract_features(model=model,tokenizer=tokenizer,device=device,ds=ds_train).cpu().detach().numpy()
cls_feats_val = extract_features(model=model,tokenizer=tokenizer,device=device,ds=ds_val).cpu().detach().numpy()
cls_feats_test = extract_features(model=model,tokenizer=tokenizer,device=device,ds=ds_test).cpu().detach().numpy()

label_train = np.array(ds_train['label'])
label_val = np.array(ds_val['label'])
label_test = np.array(ds_test['label'])

svc = SVC(kernel='linear')
fitted_svc = svc.fit(cls_feats_train,label_train)

In [43]:
print(classification_report(label_val,svc.predict(cls_feats_val)))

              precision    recall  f1-score   support

           0       0.80      0.85      0.82       533
           1       0.84      0.79      0.81       533

    accuracy                           0.82      1066
   macro avg       0.82      0.82      0.82      1066
weighted avg       0.82      0.82      0.82      1066



In [45]:
print(classification_report(label_test,svc.predict(cls_feats_test)))

              precision    recall  f1-score   support

           0       0.80      0.82      0.81       533
           1       0.82      0.79      0.80       533

    accuracy                           0.81      1066
   macro avg       0.81      0.81      0.81      1066
weighted avg       0.81      0.81      0.81      1066



---
---
## Exercise 2: Fine-tuning DistilBERT

In this exercise we will fine-tune the DistilBERT model to (hopefully) improve sentiment analysis performance.


---
### Exercise 2.1: Token Preprocessing

The first thing we need to do is *tokenize* our dataset splits -- we don't want to re-tokenize our inputs for every batch! Our current datasets return a dictionary with *strings*, but we want *input token ids* (i.e. the output of the tokenizer). This is easy enough to do by hand, but the Hugging Face `Dataset` class provides convenient, efficient, and *lazy* methods. See the documentation for [`Dataset.map`](https://huggingface.co/docs/datasets/v3.5.0/en/package_reference/main_classes#datasets.Dataset.map).

**Tip**: Verify that your new datasets are returning for every element: `text`, `label`, `intput_ids`, and `attention_mask`.

In [49]:
# 1. Definisci la funzione che applicheremo a ogni "blocco" di dati
def tokenize_function(ds):
        return tokenizer(ds["text"], truncation=True, padding=True,return_tenors='pt')

tokenized_ds_train = ds_train.map(tokenize_function, batched=True)
print(tokenized_ds_train.column_names)

tokenized_ds_val = ds_val.map(tokenize_function,batched = True)
tokenized_ds_val = ds_test.map(tokenize_function,batched = True)

['text', 'label', 'input_ids', 'token_type_ids', 'attention_mask']


Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]


---
### Exercise 2.2: Setting up the Model to be Fine-tuned

In this exercise we need to prepare the base Distilbert model for fine-tuning for a *sequence classification task*. This means, at the very least, appending a new, randomly-initialized classification head connected to the `[CLS]` token of the last transformer layer. Luckily, HuggingFace already provides an `AutoModel` for just this type of instantiation: [`AutoModelForSequenceClassification`](https://huggingface.co/transformers/v3.0.2/model_doc/auto.html#automodelforsequenceclassification). You will want you instantiate one of these for fine-tuning.

In [1]:
from transformers import DistilBertForSequenceClassification

cls_model = DistilBertForSequenceClassification.from_pretrained('distilbert/distilbert-base-uncased', num_labels=2) 

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
cls_model

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
          (ffn): FFN(
            (dropout): Dropout(

Now let's inspect the output of the model

In [4]:
tokens = tokenizer(ds_train[2]['text'], return_tensors='pt', padding='max_length', max_length=512)
cls_model.forward(tokens['input_ids'])

NameError: name 'tokenizer' is not defined


---
### Exercise 2.3: Fine-tuning DistilBERT

Finally. In this exercise you should use a HuggingFace [`Trainer`](https://huggingface.co/docs/transformers/main/en/trainer) to fine-tune your model on the Rotten Tomatoes training split. Setting up the trainer will involve (at least):


1. Instantiating a [`DataCollatorWithPadding`](https://huggingface.co/docs/transformers/en/main_classes/data_collator) object which is what *actually* does your batch construction (by padding all sequences to the same length).
2. Writing an *evaluation function* that will measure the classification accuracy. This function takes a single argument which is a tuple containing `(logits, labels)` which you should use to compute classification accuracy (and maybe other metrics like F1 score, precision, recall) and return a `dict` with these metrics.  
3. Instantiating a [`TrainingArguments`](https://huggingface.co/docs/transformers/v4.51.1/en/main_classes/trainer#transformers.TrainingArguments) object using some reasonable defaults.
4. Instantiating a `Trainer` object using your train and validation splits, you data collator, and function to compute performance metrics.
5. Calling `trainer.train()`, waiting, waiting some more, and then calling `trainer.evaluate()` to see how it did.

**Tip**: When prototyping this laboratory I discovered the HuggingFace [Evaluate library](https://huggingface.co/docs/evaluate/en/index) which provides evaluation metrics. However I found it to have insufferable layers of abstraction and getting actual metrics computed. I suggest just using the Scikit-learn metrics...

In [28]:
# Your code here.


---
---
## Exercise 3: Choose your Own Adventure

As promised, you should choose **one** of the following exercises to work. Well, at *least* one. If you want to do them all, that is also OK! Or if you want to propose something else as a third exercise, reach out to me on the Discord!


---
### Exercise 3.1: Efficient Fine-tuning for Sentiment Analysis

In Exercise 2 we fine-tuned the *entire* Distilbert model on Rotten Tomatoes. This is expensive, even for a small model. Find an *efficient* way to fine-tune Distilbert on the Rotten Tomatoes dataset (or some other dataset).

**Hint**: You could check out the [HuggingFace PEFT library](https://huggingface.co/docs/peft/en/index) for some state-of-the-art approaches that should "just work". How else might you go about making fine-tuning more efficient without having to change your training pipeline from above?

**Why choose this exercise?** PEFT techniques -- especially LoRA are the methods of choice for adapting models to new tasks.

In [29]:
# Your code here.


---
### Exercise 3.2: Fine-tuning a CLIP Model (harder)

Use a (small) CLIP model like [`openai/clip-vit-base-patch16`](https://huggingface.co/openai/clip-vit-base-patch16) and evaluate its zero-shot performance on a small image classification dataset like ImageNette or TinyImageNet. Fine-tune (using a parameter-efficient method!) the CLIP model to see how much improvement you can squeeze out of it.

**Note**: There are several ways to adapt the CLIP model; you could fine-tune the image encoder, the text encoder, or both. Or, you could experiment with prompt learning.

**Tip**: CLIP probably already works very well on ImageNet and ImageNet-like images. For extra fun, look for an image classification dataset with different image types (e.g. *sketches*).

**Why choose this exercise?** CLIP is probably the most widely used Vision-Language Model, and adapting it is a useful skill to master.

In [30]:
# Your code here.


---
### Exercise 3.3: A Text-to-image Retrieval System (hard, but not *too* hard)

Implement a simple text-to-image retrieval system with a simple user interface --- using, for example, [gradio](https://www.gradio.app/), or [Marimo](https://marimo.io/), or [Shiny](https://shiny.posit.co/). Your application should *index* (e.g. compute visual descriptors for) a small dataset of images like [Flickr8k](https://huggingface.co/datasets/jxie/flickr8k). It should provide a user interface with which a user can enter a short text prompt (e.g. "a photo of dogs playing in the snow") and then display the top-10 matching images from the indexed dataset.

Note that there is no following code block with "Your code here" for this exercise. You will definitely want to implement this outside of a Jupyter Notebook.

**Hint**: The **CLIP** model is practically *made* for just such an application.

**Why choose this exercise?** Well, this is a course on Deep Learning *Applications*, and this is your chance to *build* one!

---
---